In [1]:
import pandas as pd
from pathlib import Path
from src.config import settings
import os

## CARREGAMENTO DAS BASES

In [16]:
CENSO_PATH          = settings.EXTERNAL_DATA_PATH / 'censo_demografico_2022.csv'
CENSO_PCD_PATH      = settings.EXTERNAL_DATA_PATH / 'censo_demografico_pcd_2022.csv'
TABELA_SNIIC        = settings.FINAL_DATA_PATH / 'tabela_final_3_2_2026_14_22.parquet'

In [ ]:
# Censo
df_censo        = pd.read_csv(CENSO_PATH, sep=';')
df_censo_pcd    = pd.read_csv(CENSO_PCD_PATH, sep=';')
df_censo_pcd.rename(columns={'Cód.':'cod_ibge', 'Total':'pop_pcd'}, inplace=True)
# SNIIC
df_sniic = pd.read_parquet(TABELA_SNIIC)

,ente_federativo,nome_pdf_pk,valor_total,perc_cotas_negras,perc_cotas_indigenas,perc_cotas_pcd,vagas_totais,específico,flag_cotas_negras,flag_cotas_indigenas,...,cod_ibge,tipo_ente,vagas_cotas_negras,vagas_cotas_indigenas,vagas_cotas_pcd,valor_por_vaga,valor_cotas_negras,valor_cotas_indigenas,valor_cotas_pcd,tipo_edital
0,ACRE,ACRE_BOLSA_112024.pdf,696000.00,0.2600,0.1200,0.1200,42,None,True,True,...,12,ESTADO,11,5,5,16571.428571,1.822857e+05,82857.142857,82857.142857,BOLSA
1,ACRE,ACRE_CULTURAVIVA_012025.pdf,1470000.00,0.2857,0.1429,0.1429,42,None,True,True,...,12,ESTADO,12,6,6,35000.000000,4.200000e+05,210000.000000,210000.000000,CULTURA VIVA
2,ACRE,ACRE_CULTURAVIVA_122024.pdf,1672276.38,0.1667,0.1667,0.1667,30,None,False,True,...,12,ESTADO,5,5,5,55742.546000,2.787127e+05,278712.730000,278712.730000,CULTURA VIVA
3,ACRE,ACRE_CULTURAVIVA_132024.pdf,418069.10,0.0000,0.0000,0.0000,1,None,True,True,...,12,ESTADO,0,0,0,418069.100000,0.000000e+00,0.000000,0.000000,CULTURA VIVA
4,ACRE,ACRE_FOMENTO_062024.pdf,4100000.00,0.2500,0.1800,0.1800,122,None,True,True,...,12,ESTADO,30,22,22,33606.557377,1.008197e+06,739344.262295,739344.262295,FOMENTO


## CENSO DEMOGRÁFICO

In [4]:
df_censo = df_censo.merge(
    right=df_censo_pcd[['cod_ibge', 'pop_pcd']],
    how='left',
    on='cod_ibge'   
)

In [5]:
# RENOMEIA AS COLUNAS DA TABELA ORIGEM
df_censo = df_censo.rename(columns=settings.RENAME_COLUMNS_CENSO)

In [6]:
# DEFINE COLUNA PESSOAS NEGRAS
df_censo["pop_pessoas_negras"] = df_censo["pop_preta"] + df_censo["pop_parda"]

In [8]:
# CRIA A COLUNA PERCENTUAL PARA CADA CATEGORIA DE POPULAÇÃO
df_censo[settings.COLS_NUM] = df_censo[settings.COLS_NUM].apply(
    pd.to_numeric, errors="coerce"
)

for col in settings.COLS_POP:
    df_censo[f"rel_{col}"] = df_censo[col] / df_censo["pop_total"]

In [11]:
# PASSA COD_IBGE PARA STRING, ESSA SERÁ NOSSA CHAVE PRIMÁRIA
df_censo['cod_ibge'] = df_censo['cod_ibge'].astype(str) 

## TABELA SNIIC